# 06 GLASS Interactive SI Shift

Display interactive latest-observed-count vs future SI-shift plots, with GLASS pathogen-antibiotic pairs highlighted.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipelines import (
    build_glass_match_table,
    build_novel_prediction_table,
    build_si_analysis_paths,
    load_or_build_interactive_rank,
    load_si_and_clean_glass_list,
    plot_future_si_vs_si_shift,
    plot_latest_si_vs_future_si,
    summarize_pathogen_coverage,
)
from amr.si import plot_interactive_latest_observed_count_vs_future_si_shift
from amr.pipeline_config import cfg, cfg_path, load_pipeline_config


## Parameters

In [3]:
PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)

OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.output_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.parquet_dir", PROJECT_ROOT)
FIGURE_OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.figure_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PARQUET_OUTPUT_DIR
ASSETS_DIR = cfg_path(PIPELINE_CONFIG, "paths.assets_dir", PROJECT_ROOT)

ANTIBIOTIC_ANALYSIS_LEVEL = cfg(PIPELINE_CONFIG, "glass_interactive.antibiotic_analysis_level")
SI_PATHS = build_si_analysis_paths(PROJECT_ROOT, ANTIBIOTIC_ANALYSIS_LEVEL)
YEARLY_SCREENED_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_screened", PROJECT_ROOT)
YEARLY_SI_PATH = SI_PATHS["yearly_si"]
RANK_OUTPUT_PATH = SI_PATHS["rank_output"]
INTERESTING_OUTPUT_PATH = SI_PATHS["interesting_output"]

GLASS_XLSX_PATH = cfg_path(PIPELINE_CONFIG, "paths.glass_xlsx", PROJECT_ROOT)
GLASS_CLEANED_PATH = cfg_path(PIPELINE_CONFIG, "paths.glass_cleaned", PROJECT_ROOT)
BACTERIA_SYNONYM_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogens", PROJECT_ROOT)
ANTIBIOTIC_SYNONYM_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_synonyms", PROJECT_ROOT)
ANTIBIOTIC_COARSENING_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_coarsening", PROJECT_ROOT)

# Reuse SI fit window settings so interactive plots stay aligned with notebook 05 outputs.
FIT_START_YEAR = cfg(PIPELINE_CONFIG, "si.fit_start_year")
FIT_END_YEAR = cfg(PIPELINE_CONFIG, "si.fit_end_year")
CURVE_FIT_END_YEAR = cfg(PIPELINE_CONFIG, "si.curve_fit_end_year")
NUM_FUTURE_YEARS = cfg(PIPELINE_CONFIG, "glass_interactive.num_future_years")
LOESS_FRAC = cfg(PIPELINE_CONFIG, "si.loess_frac")
LOESS_DEGREE = cfg(PIPELINE_CONFIG, "si.loess_degree")
TAIL_DAMPING_YEARS = cfg(PIPELINE_CONFIG, "si.tail_damping_years")
MIN_AMR_PAIR_PUBLICATION_COUNT = cfg(PIPELINE_CONFIG, "si.min_amr_pair_publication_count")
MIN_NUM_YEARS_WITH_VALID_COUNTS = cfg(PIPELINE_CONFIG, "si.min_num_years_with_valid_counts")

# Interactive filtering and plot layout controls.
MIN_LAST_AVAILABLE_COUNT = cfg(PIPELINE_CONFIG, "glass_interactive.min_last_available_count")
MIN_TOTAL_PUBLICATIONS = cfg(PIPELINE_CONFIG, "glass_interactive.min_total_publications")
MIN_COUNT_ABS_PCT_ERROR_MEAN = cfg(PIPELINE_CONFIG, "glass_interactive.min_count_abs_pct_error_mean")
MAX_COUNT_ABS_PCT_ERROR_MEAN = cfg(PIPELINE_CONFIG, "glass_interactive.max_count_abs_pct_error_mean")
MIN_COUNTS_BACKGROUND = cfg(PIPELINE_CONFIG, "glass_interactive.min_counts_background")
MIN_ABS_DELTA_SI = cfg(PIPELINE_CONFIG, "glass_interactive.min_abs_delta_si")
MIN_SI_DELTA = cfg(PIPELINE_CONFIG, "glass_interactive.min_si_delta")
DISPLAY_NAME = cfg(PIPELINE_CONFIG, "glass_interactive.display_name")
MAX_TEXT_LABELS = cfg(PIPELINE_CONFIG, "glass_interactive.max_text_labels")
WIDTH = cfg(PIPELINE_CONFIG, "glass_interactive.width")
HEIGHT = cfg(PIPELINE_CONFIG, "glass_interactive.height")

if not YEARLY_SCREENED_PATH.exists():
    raise FileNotFoundError(f"Missing {YEARLY_SCREENED_PATH}. Run notebook 04 to create screened AMR pairs first.")
if not YEARLY_SI_PATH.exists():
    raise FileNotFoundError(f"Missing {YEARLY_SI_PATH}. Run notebook 05 after notebook 04 to build SI outputs.")
if YEARLY_SI_PATH.stat().st_mtime < YEARLY_SCREENED_PATH.stat().st_mtime:
    raise RuntimeError(
        f"{YEARLY_SI_PATH} is older than {YEARLY_SCREENED_PATH}. "
        "Run notebook 05 after the latest notebook 04 LLM/screened-pair run."
    )


## Load SI And Clean GLASS List

In [4]:
si_df, glass_pairs_df = load_si_and_clean_glass_list(
    YEARLY_SI_PATH,
    GLASS_XLSX_PATH,
    bacteria_synonym_path=BACTERIA_SYNONYM_PATH,
    antibiotic_synonym_path=ANTIBIOTIC_SYNONYM_PATH,
    antibiotic_coarsening_path=ANTIBIOTIC_COARSENING_PATH,
    glass_cleaned_path=GLASS_CLEANED_PATH,
)

si_source_df = pd.read_parquet(YEARLY_SCREENED_PATH)
si_pathogen_summary_df = summarize_pathogen_coverage(
    source_df=si_source_df,
    modeled_df=si_df,
    source_label="screened AMR input from notebook 04",
    modeled_label="filtered SI dataset from notebook 05",
)
display(si_pathogen_summary_df.query("status != 'present_in_both'"))
display(glass_pairs_df.head(20))

Loaded SI rows: 4,605 from /Users/f.costa/Code/Antimicrobial-Resistance/outputs/parquets/yearly_si.parquet
Pathogens: 26; antibiotics/levels: 56
Saved cleaned GLASS rows: 89 to /Users/f.costa/Code/Antimicrobial-Resistance/assets/glass_amr_cleaned.csv
screened AMR input from notebook 04 pathogens: 201
filtered SI dataset from notebook 05 pathogens: 26
Pathogens present in screened AMR input from notebook 04 but absent from filtered SI dataset from notebook 05: 175
Achromobacter denitrificans, Achromobacter xylosoxidans, Acinetobacter calcoaceticus, Actinotignum schaalii, Aeromonas caviae, Aeromonas hydrophila, Aeromonas salmonicida, Aeromonas sobria, Aeromonas veronii, Aggregatibacter actinomycetemcomitans, Bacillus anthracis, Bacillus cereus, Bartonella bacilliformis, Bartonella quintana, Bordetella bronchiseptica, Bordetella hinzii, Bordetella parapertussis, Bordetella pertussis, Borrelia burgdorferi, Brucella intermedia, Brucella melitensis, Burkholderia cenocepacia, Burkholderia cep

,pathogen,source_rows,source_publications,source_pairs,modeled_rows,modeled_publications,modeled_pairs,in_source,in_modeled,status
0,Achromobacter denitrificans,20,21,12,0.0,0.0,0.0,True,False,missing_from_modeled
1,Achromobacter xylosoxidans,44,50,24,0.0,0.0,0.0,True,False,missing_from_modeled
2,Acinetobacter calcoaceticus,35,35,19,0.0,0.0,0.0,True,False,missing_from_modeled
3,Actinotignum schaalii,11,11,11,0.0,0.0,0.0,True,False,missing_from_modeled
4,Aeromonas caviae,32,35,22,0.0,0.0,0.0,True,False,missing_from_modeled
...,...,...,...,...,...,...,...,...,...,...
170,Vibrio mimicus,1,1,1,0.0,0.0,0.0,True,False,missing_from_modeled
171,Vibrio vulnificus,28,28,16,0.0,0.0,0.0,True,False,missing_from_modeled
172,Yersinia enterocolitica,251,136,50,0.0,0.0,0.0,True,False,missing_from_modeled
173,Yersinia pestis,19,10,9,0.0,0.0,0.0,True,False,missing_from_modeled


,raw_pathogen,raw_antibiotic,pathogen,antibiotic,pathogen_standardized,antibiotic_standardized,Class,Simplified categories,High level categories,Origin
0,Acinetobacter spp.,amikacin,Acinetobacter baumannii,Amikacin,True,True,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
1,Acinetobacter spp.,Colistin,Acinetobacter baumannii,Colistin,True,False,Polymyxins,Miscellaneous,Other,Naturally occurring/modified
2,Acinetobacter spp.,doripenem,Acinetobacter baumannii,Doripenem,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
3,Acinetobacter spp.,Gentamicin,Acinetobacter baumannii,Gentamicin,True,False,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
4,Acinetobacter spp.,Imipenem,Acinetobacter baumannii,Imipenem/cilastatin,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
5,Acinetobacter spp.,meropenem,Acinetobacter baumannii,Meropenem,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
6,Acinetobacter spp.,minocycline,Acinetobacter baumannii,Minocycline,True,True,Tetracyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
7,Acinetobacter spp.,Tigecycline,Acinetobacter baumannii,Tigecycline,True,False,Glycylcyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
8,E. coli,Cefepime,Escherichia coli,Cefepime,True,False,Fourth-generation-cephalosporins,Cephalosporins,Beta lactam and related,Naturally occurring/modified
9,E. coli,cefotaxime,Escherichia coli,Cefotaxime,True,True,Third-generation-cephalosporins,Cephalosporins,Beta lactam and related,Naturally occurring/modified


## Build Or Load Rank Data

In [5]:
rank_df = load_or_build_interactive_rank(
    si_df,
    RANK_OUTPUT_PATH,
    source_path=YEARLY_SI_PATH,
    fit_start_year=FIT_START_YEAR,
    fit_end_year=FIT_END_YEAR,
    curve_fit_end_year=CURVE_FIT_END_YEAR,
    num_future_years=NUM_FUTURE_YEARS,
    loess_frac=LOESS_FRAC,
    loess_degree=LOESS_DEGREE,
    tail_damping_years=TAIL_DAMPING_YEARS,
    min_amr_pair_publication_count=MIN_AMR_PAIR_PUBLICATION_COUNT,
    min_num_years_with_valid_counts=MIN_NUM_YEARS_WITH_VALID_COUNTS,
    verbose=True,
)

if not INTERESTING_OUTPUT_PATH.exists():
    raise FileNotFoundError(
        f"Missing {INTERESTING_OUTPUT_PATH}. Run notebook 05 first so only pairs satisfying all SI criteria enter the GLASS analysis."
    )
if INTERESTING_OUTPUT_PATH.stat().st_mtime < YEARLY_SI_PATH.stat().st_mtime:
    raise RuntimeError(
        f"{INTERESTING_OUTPUT_PATH} is older than {YEARLY_SI_PATH}. Rerun notebook 05 so its configured SI eligibility criteria are applied."
    )
eligible_pairs = pd.read_parquet(INTERESTING_OUTPUT_PATH)[["pathogen", "antibiotic"]].drop_duplicates()
rank_df = rank_df.merge(eligible_pairs, on=["pathogen", "antibiotic"], how="inner", validate="one_to_one")
print(f"Eligible SI pairs entering Notebook 06: {len(rank_df):,}")
display(rank_df)

Recomputing rank data because cached fit metadata does not match the requested endpoints.
[1/187] ranking Acinetobacter baumannii vs Amikacin
[2/187] ranking Acinetobacter baumannii vs Ceftazidime
[3/187] ranking Acinetobacter baumannii vs Ciprofloxacin
[4/187] ranking Acinetobacter baumannii vs Colistin
[5/187] ranking Acinetobacter baumannii vs Gentamicin
[6/187] ranking Acinetobacter baumannii vs Meropenem
[7/187] ranking Acinetobacter baumannii vs Polymyxin-B
[8/187] ranking Acinetobacter baumannii vs Sulbactam
[9/187] ranking Acinetobacter baumannii vs Tetracycline
[10/187] ranking Acinetobacter baumannii vs Tigecycline
[11/187] ranking Bacteroides fragilis vs Metronidazole
[12/187] ranking Campylobacter coli vs Ciprofloxacin
[13/187] ranking Campylobacter coli vs Erythromycin
[14/187] ranking Campylobacter coli vs Tetracycline
[15/187] ranking Campylobacter jejuni vs Ampicillin
[16/187] ranking Campylobacter jejuni vs Ciprofloxacin
[17/187] ranking Campylobacter jejuni vs Erythro

,pathogen,antibiotic,reference_year,future_year,num_future_years,fit_start_year,fit_end_year,curve_fit_end_year,post_fit_validation_end_year,raw_latest_available_year,...,tail_damping_years,validation_loess_frac,validation_loess_degree,validation_tail_damping_years,status,error,current_rank,future_rank,rank_change,abs_rank_change
0,Escherichia coli,Piperacillin,2025,2028,3,2000,2023,2025,2025,2026.0,...,NaN,0.6,1,NaN,ok,NaN,82.0,70.0,12.0,12.0
1,Mycobacterium tuberculosis,Rifampicin,2025,2028,3,2000,2023,2025,2025,2026.0,...,NaN,0.6,1,NaN,ok,NaN,22.0,13.0,9.0,9.0
2,Pseudomonas aeruginosa,Piperacillin,2025,2028,3,2000,2023,2025,2025,2026.0,...,NaN,0.6,1,NaN,ok,NaN,29.0,20.0,9.0,9.0
3,Escherichia coli,Ceftriaxone,2025,2028,3,2000,2023,2025,2025,2026.0,...,NaN,0.6,1,NaN,ok,NaN,42.0,34.0,8.0,8.0
4,Staphylococcus aureus,Ampicillin,2025,2028,3,2000,2023,2025,2025,2026.0,...,NaN,0.6,1,NaN,ok,NaN,35.0,28.0,7.0,7.0
5,Staphylococcus aureus,Daptomycin,2025,2028,3,2000,2023,2025,2025,2026.0,...,NaN,0.6,1,NaN,ok,NaN,36.0,29.0,7.0,7.0
6,Klebsiella pneumoniae,Meropenem,2025,2028,3,2000,2023,2025,2025,2026.0,...,NaN,0.6,1,NaN,ok,NaN,8.0,3.0,5.0,5.0
7,Helicobacter pylori,Levofloxacin,2025,2028,3,2000,2023,2025,2025,2026.0,...,NaN,0.6,1,NaN,ok,NaN,4.0,8.0,-4.0,4.0
8,Enterococcus faecium,Vancomycin,2025,2028,3,2000,2023,2025,2025,2026.0,...,NaN,0.6,1,NaN,ok,NaN,10.0,7.0,3.0,3.0
9,Escherichia coli,Cefotaxime,2025,2028,3,2000,2023,2025,2025,2026.0,...,NaN,0.6,1,NaN,ok,NaN,7.0,10.0,-3.0,3.0


## Interactive Latest Count Vs Future SI Shift

In [6]:
latest_count_shift_result = plot_interactive_latest_observed_count_vs_future_si_shift(
    rank_df,
    glass_pairs_df=glass_pairs_df,
    min_last_available_count=MIN_LAST_AVAILABLE_COUNT,
    min_total_publications=MIN_TOTAL_PUBLICATIONS,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    min_counts_background=MIN_COUNTS_BACKGROUND,
    min_abs_delta_si=MIN_ABS_DELTA_SI,
    min_si_delta=MIN_SI_DELTA,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    width=WIDTH,
    height=HEIGHT,
    pdf_path=FIGURE_OUTPUT_DIR / "interactive_latest_observed_count_vs_future_si_shift.pdf",
    show=True,
)
latest_count_shift_df = latest_count_shift_result["plot_df"]
print(f"Displayed rows: {len(latest_count_shift_df):,}; GLASS matches: {int(latest_count_shift_df['is_glass_pair'].sum()):,}")
if latest_count_shift_result["pdf_path"]:
    print(f"Saved PDF: {latest_count_shift_result['pdf_path']}")
elif latest_count_shift_result.get("pdf_error"):
    print(f"PDF export failed: {latest_count_shift_result['pdf_error']}")

Displayed rows: 25; GLASS matches: 8
Saved PDF: /Users/f.costa/Code/Antimicrobial-Resistance/outputs/figures/interactive_latest_observed_count_vs_future_si_shift.pdf


## Display GLASS Matches In Plot

In [7]:
glass_plot_matches_df = build_glass_match_table(latest_count_shift_df)
display(glass_plot_matches_df)

,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,total_publications,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean,glass_group_column
0,Klebsiella pneumoniae,Colistin,2025.0,2028,58.0,378.0,105.211412,0.422676,0.587011,0.164336,8.594378,antibiotic
1,Klebsiella pneumoniae,Meropenem,2025.0,2028,48.0,325.0,104.412999,0.407314,0.592114,0.184800,24.669736,antibiotic
2,Acinetobacter baumannii,Colistin,2025.0,2028,44.0,354.0,44.274628,0.227194,0.267055,0.039861,21.928050,antibiotic
3,Staphylococcus aureus,Oxacillin,2025.0,2028,44.0,550.0,56.488618,0.207144,0.253670,0.046526,22.710845,antibiotic
4,Klebsiella pneumoniae,Ceftazidime,2025.0,2028,36.0,227.0,106.679037,0.485337,0.764380,0.279043,53.552346,antibiotic
5,Pseudomonas aeruginosa,Meropenem,2025.0,2028,36.0,339.0,46.523677,0.104403,0.145002,0.040600,4.721095,antibiotic
6,Salmonella enterica,Ciprofloxacin,2025.0,2028,28.0,408.0,38.767589,0.293803,0.323394,0.029592,47.185146,antibiotic
7,Escherichia coli,Ceftriaxone,2025.0,2028,27.0,249.0,36.990928,0.122138,0.217809,0.095671,4.189418,antibiotic


## Novel Non-GLASS Predictions

These are plotted pairs that do not appear in the cleaned GLASS list, sorted as candidate novel predictions of interest.

In [8]:
novel_prediction_df = build_novel_prediction_table(latest_count_shift_df)
print(f"Novel non-GLASS predictions displayed: {len(novel_prediction_df):,}")
display(novel_prediction_df)

Novel non-GLASS predictions displayed: 17


,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,total_publications,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean
0,Mycobacterium tuberculosis,Rifampicin,2025.0,2028,22.0,172.0,34.505166,0.245301,0.405569,0.160268,5.144846
1,Staphylococcus aureus,Cefoxitin,2025.0,2028,37.0,238.0,49.699241,0.470495,0.591378,0.120884,10.806106
2,Staphylococcus aureus,Daptomycin,2025.0,2028,16.0,147.0,19.715495,0.144017,0.254433,0.110417,14.058187
3,Pseudomonas aeruginosa,Piperacillin,2025.0,2028,21.0,174.0,27.964535,0.203456,0.312406,0.108950,21.253969
4,Staphylococcus aureus,Ampicillin,2025.0,2028,36.0,269.0,48.172466,0.153195,0.261942,0.108747,19.822006
5,Staphylococcus aureus,Meticillin,2025.0,2028,470.0,3843.0,564.190322,0.420761,0.518283,0.097522,10.405435
6,Enterococcus faecium,Vancomycin,2025.0,2028,56.0,477.0,67.581741,0.385699,0.462392,0.076693,15.162455
7,Escherichia coli,Amoxicillin,2025.0,2028,50.0,349.0,46.400931,0.289055,0.364242,0.075187,34.139775
8,Staphylococcus aureus,Vancomycin,2025.0,2028,98.0,928.0,109.913961,0.358266,0.425271,0.067006,35.867786
9,Staphylococcus aureus,Linezolid,2025.0,2028,34.0,354.0,49.357025,0.204687,0.267444,0.062757,19.566942


## Future SI Vs Future SI Shift

This uses the same y-axis definition as the latest-count plot: predicted future SI minus smoothed SI at the latest available year.

In [9]:
future_si_shift_result = plot_future_si_vs_si_shift(
    latest_count_shift_df,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    width=WIDTH,
    height=HEIGHT,
    pdf_path=FIGURE_OUTPUT_DIR / "future_si_vs_si_shift.pdf",
    show=True,
)
future_si_shift_fig = future_si_shift_result["figure"]
if future_si_shift_result["pdf_path"]:
    print(f"Saved PDF: {future_si_shift_result['pdf_path']}")
elif future_si_shift_result.get("pdf_error"):
    print(f"PDF export failed: {future_si_shift_result['pdf_error']}")

Saved PDF: /Users/f.costa/Code/Antimicrobial-Resistance/outputs/figures/future_si_vs_si_shift.pdf


In [10]:
latest_count_shift_df.to_csv(
    FIGURE_OUTPUT_DIR / "latest_count_shift_df.csv",
    index=False,
)